In [ ]:
"""
Data ingestion and SQLite star-schema loader
for the Mutual Fund Analytics project.

Fixes:
1. Builds dim_date from ALL relevant source dates.
2. Loads fact_nav using fund_key + date_key.
3. Loads fact_aum from scheme-level AUM in
   07_scheme_performance.csv.
"""

from pathlib import Path
import sqlite3
import pandas as pd


# -------------------------------------------------------------------
# Project paths
# -------------------------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent.parent
RAW_DIR = BASE_DIR / "data" / "raw"
DATABASE_DIR = BASE_DIR / "database"
DB_PATH = DATABASE_DIR / "bluestock_mf.db"


# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------

def read_csv(filename: str) -> pd.DataFrame:
    """Read a CSV file from data/raw."""
    path = RAW_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    return pd.read_csv(path)


def create_tables(conn: sqlite3.Connection) -> None:
    """Create the required star-schema tables."""

    cursor = conn.cursor()

    # Drop tables in dependency order
    cursor.executescript(
        """
        DROP TABLE IF EXISTS fact_transactions;
        DROP TABLE IF EXISTS fact_performance;
        DROP TABLE IF EXISTS fact_aum;
        DROP TABLE IF EXISTS fact_nav;
        DROP TABLE IF EXISTS dim_date;
        DROP TABLE IF EXISTS dim_fund;
        """
    )

    # ---------------------------------------------------------------
    # dim_fund
    # ---------------------------------------------------------------

    cursor.execute(
        """
        CREATE TABLE dim_fund (
            fund_key INTEGER PRIMARY KEY AUTOINCREMENT,
            amfi_code INTEGER UNIQUE,
            fund_house TEXT,
            scheme_name TEXT,
            category TEXT,
            sub_category TEXT,
            plan TEXT,
            launch_date TEXT,
            benchmark TEXT,
            expense_ratio_pct REAL,
            exit_load_pct REAL,
            min_sip_amount REAL,
            min_lumpsum_amount REAL,
            fund_manager TEXT,
            risk_category TEXT,
            sebi_category_code TEXT
        )
        """
    )

    # ---------------------------------------------------------------
    # dim_date
    # ---------------------------------------------------------------

    cursor.execute(
        """
        CREATE TABLE dim_date (
            date_key INTEGER PRIMARY KEY,
            date TEXT UNIQUE,
            year INTEGER,
            month INTEGER,
            quarter INTEGER,
            day INTEGER
        )
        """
    )

    # ---------------------------------------------------------------
    # fact_nav
    # ---------------------------------------------------------------

    cursor.execute(
        """
        CREATE TABLE fact_nav (
            nav_id INTEGER PRIMARY KEY AUTOINCREMENT,
            fund_key INTEGER,
            date_key INTEGER,
            nav REAL,
            FOREIGN KEY(fund_key)
                REFERENCES dim_fund(fund_key),
            FOREIGN KEY(date_key)
                REFERENCES dim_date(date_key)
        )
        """
    )

    # ---------------------------------------------------------------
    # fact_aum
    # ---------------------------------------------------------------

    cursor.execute(
        """
        CREATE TABLE fact_aum (
            aum_key INTEGER PRIMARY KEY AUTOINCREMENT,
            fund_key INTEGER,
            date_key INTEGER,
            aum_crore REAL,
            FOREIGN KEY(fund_key)
                REFERENCES dim_fund(fund_key),
            FOREIGN KEY(date_key)
                REFERENCES dim_date(date_key)
        )
        """
    )

    # ---------------------------------------------------------------
    # fact_performance
    # ---------------------------------------------------------------

    cursor.execute(
        """
        CREATE TABLE fact_performance (
            performance_key INTEGER PRIMARY KEY AUTOINCREMENT,
            fund_key INTEGER,
            return_1yr_pct REAL,
            return_3yr_pct REAL,
            return_5yr_pct REAL,
            benchmark_3yr_pct REAL,
            alpha REAL,
            beta REAL,
            sharpe_ratio REAL,
            sortino_ratio REAL,
            std_dev_ann_pct REAL,
            max_drawdown_pct REAL,
            FOREIGN KEY(fund_key)
                REFERENCES dim_fund(fund_key)
        )
        """
    )

    # ---------------------------------------------------------------
    # fact_transactions
    # ---------------------------------------------------------------

    cursor.execute(
        """
        CREATE TABLE fact_transactions (
            transaction_key INTEGER PRIMARY KEY AUTOINCREMENT,
            investor_id INTEGER,
            fund_key INTEGER,
            date_key INTEGER,
            transaction_type TEXT,
            amount_inr REAL,
            kyc_status TEXT,
            state TEXT,
            FOREIGN KEY(fund_key)
                REFERENCES dim_fund(fund_key),
            FOREIGN KEY(date_key)
                REFERENCES dim_date(date_key)
        )
        """
    )

    conn.commit()


# -------------------------------------------------------------------
# Main ETL
# -------------------------------------------------------------------

def main():
    """Run complete data ingestion and star-schema loading."""

    DATABASE_DIR.mkdir(parents=True, exist_ok=True)

    # ---------------------------------------------------------------
    # Read source files
    # ---------------------------------------------------------------

    fund = read_csv("01_fund_master.csv")
    nav = read_csv("02_nav_history.csv")
    aum_house = read_csv("03_aum_by_fund_house.csv")
    performance = read_csv("07_scheme_performance.csv")
    transactions = read_csv("08_investor_transactions.csv")

    # ---------------------------------------------------------------
    # Clean dates
    # ---------------------------------------------------------------

    nav["date"] = pd.to_datetime(nav["date"], errors="coerce")
    aum_house["date"] = pd.to_datetime(
        aum_house["date"],
        errors="coerce"
    )
    transactions["transaction_date"] = pd.to_datetime(
        transactions["transaction_date"],
        errors="coerce"
    )
    fund["launch_date"] = pd.to_datetime(
        fund["launch_date"],
        errors="coerce"
    )

    # Remove invalid dates
    nav = nav.dropna(subset=["date"])
    aum_house = aum_house.dropna(subset=["date"])
    transactions = transactions.dropna(subset=["transaction_date"])

    # ---------------------------------------------------------------
    # Clean AMFI codes
    # ---------------------------------------------------------------

    fund["amfi_code"] = pd.to_numeric(
        fund["amfi_code"],
        errors="coerce"
    )

    nav["amfi_code"] = pd.to_numeric(
        nav["amfi_code"],
        errors="coerce"
    )

    performance["amfi_code"] = pd.to_numeric(
        performance["amfi_code"],
        errors="coerce"
    )

    transactions["amfi_code"] = pd.to_numeric(
        transactions["amfi_code"],
        errors="coerce"
    )

    fund = fund.dropna(subset=["amfi_code"])
    nav = nav.dropna(subset=["amfi_code"])
    performance = performance.dropna(subset=["amfi_code"])
    transactions = transactions.dropna(subset=["amfi_code"])

    fund["amfi_code"] = fund["amfi_code"].astype(int)
    nav["amfi_code"] = nav["amfi_code"].astype(int)
    performance["amfi_code"] = performance["amfi_code"].astype(int)
    transactions["amfi_code"] = transactions["amfi_code"].astype(int)

    # ---------------------------------------------------------------
    # Create fresh database
    # ---------------------------------------------------------------

    if DB_PATH.exists():
        DB_PATH.unlink()

    conn = sqlite3.connect(DB_PATH)

    try:
        create_tables(conn)

        # ===========================================================
        # 1. LOAD dim_fund
        # ===========================================================

        fund_columns = [
            "amfi_code",
            "fund_house",
            "scheme_name",
            "category",
            "sub_category",
            "plan",
            "launch_date",
            "benchmark",
            "expense_ratio_pct",
            "exit_load_pct",
            "min_sip_amount",
            "min_lumpsum_amount",
            "fund_manager",
            "risk_category",
            "sebi_category_code",
        ]

        fund_dim = fund[fund_columns].copy()

        fund_dim["launch_date"] = fund_dim[
            "launch_date"
        ].dt.strftime("%Y-%m-%d")

        fund_dim.to_sql(
            "dim_fund",
            conn,
            if_exists="append",
            index=False
        )

        # ===========================================================
        # 2. CREATE FUND KEY MAPPING
        # ===========================================================

        fund_map = pd.read_sql_query(
            """
            SELECT fund_key, amfi_code
            FROM dim_fund
            """,
            conn
        )

        # ===========================================================
        # 3. CREATE dim_date FROM ALL DATE SOURCES
        # ===========================================================

        nav_dates = nav[["date"]].rename(
            columns={"date": "full_date"}
        )

        aum_dates = aum_house[["date"]].rename(
            columns={"date": "full_date"}
        )

        transaction_dates = transactions[
            ["transaction_date"]
        ].rename(
            columns={"transaction_date": "full_date"}
        )

        all_dates = pd.concat(
            [
                nav_dates,
                aum_dates,
                transaction_dates,
            ],
            ignore_index=True
        )

        all_dates["full_date"] = pd.to_datetime(
            all_dates["full_date"],
            errors="coerce"
        )

        all_dates = (
            all_dates
            .dropna()
            .drop_duplicates()
            .sort_values("full_date")
            .reset_index(drop=True)
        )

        all_dates["date_key"] = (
            all_dates["full_date"]
            .dt.strftime("%Y%m%d")
            .astype(int)
        )

        date_dim = pd.DataFrame(
            {
                "date_key": all_dates["date_key"],
                "date": all_dates["full_date"].dt.strftime(
                    "%Y-%m-%d"
                ),
                "year": all_dates["full_date"].dt.year,
                "month": all_dates["full_date"].dt.month,
                "quarter": all_dates["full_date"].dt.quarter,
                "day": all_dates["full_date"].dt.day,
            }
        )

        date_dim.to_sql(
            "dim_date",
            conn,
            if_exists="append",
            index=False
        )

        # Date lookup
        date_map = pd.read_sql_query(
            """
            SELECT date_key, date
            FROM dim_date
            """,
            conn
        )

        date_map["date"] = pd.to_datetime(date_map["date"])

        # ===========================================================
        # 4. LOAD fact_nav
        # ===========================================================

        fact_nav = nav.merge(
            fund_map,
            on="amfi_code",
            how="inner"
        )

        fact_nav = fact_nav.merge(
            date_map,
            left_on="date",
            right_on="date",
            how="inner"
        )

        fact_nav = fact_nav[
            [
                "fund_key",
                "date_key",
                "nav",
            ]
        ].copy()

        fact_nav["nav"] = pd.to_numeric(
            fact_nav["nav"],
            errors="coerce"
        )

        fact_nav = fact_nav.dropna(
            subset=["fund_key", "date_key", "nav"]
        )

        fact_nav.to_sql(
            "fact_nav",
            conn,
            if_exists="append",
            index=False
        )

        # ===========================================================
        # 5. LOAD fact_aum
        # ===========================================================
        #
        # 07_scheme_performance has:
        # amfi_code + aum_crore
        #
        # This is fund-level AUM and therefore matches fact_aum's
        # fund_key design.
        #
        # The performance file has no date column, so we use the
        # latest AUM snapshot date available in 03_aum_by_fund_house.
        # ===========================================================

        latest_aum_date = aum_house["date"].max()

        aum_snapshot = performance[
            [
                "amfi_code",
                "aum_crore",
            ]
        ].copy()

        aum_snapshot["aum_crore"] = pd.to_numeric(
            aum_snapshot["aum_crore"],
            errors="coerce"
        )

        aum_snapshot = aum_snapshot.dropna(
            subset=["aum_crore"]
        )

        aum_snapshot = aum_snapshot.merge(
            fund_map,
            on="amfi_code",
            how="inner"
        )

        latest_date_key = int(
            latest_aum_date.strftime("%Y%m%d")
        )

        fact_aum = aum_snapshot[
            [
                "fund_key",
                "aum_crore",
            ]
        ].copy()

        fact_aum["date_key"] = latest_date_key

        fact_aum = fact_aum[
            [
                "fund_key",
                "date_key",
                "aum_crore",
            ]
        ]

        fact_aum.to_sql(
            "fact_aum",
            conn,
            if_exists="append",
            index=False
        )

        # ===========================================================
        # 6. LOAD fact_performance
        # ===========================================================

        performance = performance.merge(
            fund_map,
            on="amfi_code",
            how="inner"
        )

        performance_columns = [
            "fund_key",
            "return_1yr_pct",
            "return_3yr_pct",
            "return_5yr_pct",
            "benchmark_3yr_pct",
            "alpha",
            "beta",
            "sharpe_ratio",
            "sortino_ratio",
            "std_dev_ann_pct",
            "max_drawdown_pct",
        ]

        fact_performance = performance[
            performance_columns
        ].copy()

        fact_performance.to_sql(
            "fact_performance",
            conn,
            if_exists="append",
            index=False
        )

        # ===========================================================
        # 7. LOAD fact_transactions
        # ===========================================================

        transactions = transactions.merge(
            fund_map,
            on="amfi_code",
            how="inner"
        )

        transactions = transactions.merge(
            date_map,
            left_on="transaction_date",
            right_on="date",
            how="inner"
        )

        fact_transactions = transactions[
            [
                "investor_id",
                "fund_key",
                "date_key",
                "transaction_type",
                "amount_inr",
                "kyc_status",
                "state",
            ]
        ].copy()

        fact_transactions.to_sql(
            "fact_transactions",
            conn,
            if_exists="append",
            index=False
        )

        # ===========================================================
        # 8. VALIDATION
        # ===========================================================

        print("\nETL completed successfully.")
        print(f"Database: {DB_PATH}")

        tables = [
            "dim_fund",
            "dim_date",
            "fact_nav",
            "fact_aum",
            "fact_performance",
            "fact_transactions",
        ]

        print("\nTable row counts:")

        for table in tables:
            count = pd.read_sql_query(
                f"SELECT COUNT(*) AS count FROM {table}",
                conn
            ).iloc[0]["count"]

            print(f"{table:20s}: {count:,}")

        # Extra date validation
        date_count = pd.read_sql_query(
            """
            SELECT COUNT(DISTINCT date)
            AS unique_dates
            FROM dim_date
            """,
            conn
        ).iloc[0]["unique_dates"]

        nav_date_count = pd.read_sql_query(
            """
            SELECT COUNT(DISTINCT date_key)
            AS unique_dates
            FROM fact_nav
            """,
            conn
        ).iloc[0]["unique_dates"]

        print("\nDate validation:")
        print(f"dim_date unique dates : {date_count:,}")
        print(f"fact_nav unique dates : {nav_date_count:,}")

    finally:
        conn.close()


if __name__ == "__main__":
    main()